In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install dotenv in the CURRENT kernel's Python
import sys
!{sys.executable} -m pip install python-dotenv google-generativeai


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ================================================================
# CELL 1: IMPORTS, CONFIGURATION & SETUP
# ================================================================

import os
import sys
import time
import logging
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

from dotenv import load_dotenv
import google.generativeai as genai

# ─── LOGGING SETUP ───
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger("ProductGPT")

# ─── ENVIRONMENT SETUP ───
notebook_dir = Path.cwd()
env_path = notebook_dir / '.env.agentic'

if not env_path.exists():
    logger.error(f"❌ {env_path.name} not found in {notebook_dir}")
    sys.exit(1)

load_dotenv(dotenv_path=env_path, override=True)
api_key = os.getenv("AGENTIC_KEY")

if not api_key:
    logger.error("❌ AGENTIC_KEY not found in .env.agentic")
    sys.exit(1)

# ─── MODEL SETUP ───
genai.configure(api_key=api_key)

MODEL = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={
        "temperature": 0.7,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 1024,
    }
)

logger.info("✅ Gemini 2.5 Flash initialized")
logger.info(f"📁 Config source: {env_path.name}")

# ─── CONSTANTS ───
SYSTEM_PROMPT = """
You are ProductGPT, an expert product research assistant. 
Always communicate in a helpful, friendly 'Hinglish' tone (Hindi + English mix).

When recommending products, follow this structure:
⭐ BEST PICK — Best overall value + performance
💰 BUDGET PICK — Most affordable without compromising quality
🏆 PREMIUM PICK — High-end option with exceptional features

Include approximate prices with disclaimer: "Based on my knowledge, approximate prices..."
End product lists with: "Kya aapko inme se kisi ke detailed specs chahiye?"
Be honest — you are a knowledge assistant, not a live search engine.
"""

WELCOME_MESSAGE = "Hello! Main hoon ProductGPT — aapka expert product research assistant. Kya research karna hai aaj? 🔍"

TRACKING_KEYWORDS = [
    "laptop", "phone", "watch", "headphone", "camera", "tv",
    "tablet", "monitor", "keyboard", "mouse", "speaker",
    "iphone", "samsung", "oneplus", "macbook", "ipad", "pixel"
]

logger.info("✅ Constants loaded")
print("\n🚀 ProductGPT System Ready!\n")

C:\Users\pmthe\product-gpt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\pmthe\AppData\Local\Temp\ipykernel_19092\2736577681.py:14: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
10:13:21 | INFO     | ✅ Gemini 2.5 Flash initialized
10:13:21 | INFO     | 📁 Config source: .env.agentic
10:13:21 | INFO     | ✅ Constants loaded



🚀 ProductGPT System Ready!



In [4]:
print(f"Current directory: {Path.cwd()}")
print(f"Looking for: {Path.cwd() / '.env.agentic'}")
print(f"File exists: {(Path.cwd() / '.env.agentic').exists()}")

Current directory: C:\Users\pmthe\product-gpt
Looking for: C:\Users\pmthe\product-gpt\.env.agentic
File exists: True


In [5]:
# ================================================================
# CELL 2: AGENT CORE — STATE + THINKING ENGINE (FIXED)
# ================================================================

@dataclass
class AgentState:
    """Tracks session state across conversation turns"""
    exchange_count: int = 0
    current_topic: Optional[str] = None
    empty_input_count: int = 0
    start_time: float = field(default_factory=time.time)
    peak_tokens: int = 0

    def get_duration(self) -> str:
        d = int(time.time() - self.start_time)
        return f"{d // 60}m {d % 60}s"

    def update_topic(self, user_input: str) -> None:
        for keyword in TRACKING_KEYWORDS:
            if keyword in user_input.lower():
                self.current_topic = keyword.capitalize()
                return

    def reset(self) -> None:
        self.exchange_count = 0
        self.current_topic = None
        self.empty_input_count = 0
        self.start_time = time.time()
        self.peak_tokens = 0


def count_tokens(history: List[Dict]) -> int:
    try:
        return MODEL.count_tokens(history).total_tokens
    except Exception:
        total_chars = sum(
            len(str(msg.get("parts", [""])[0]))
            for msg in history
        )
        return total_chars // 4


def initialize_conversation() -> List[Dict]:
    return [
        {"role": "user", "parts": [SYSTEM_PROMPT]},
        {"role": "model", "parts": [WELCOME_MESSAGE]}
    ]


def agent_think(
    user_input: str,
    history: List[Dict],
    use_search: bool = False
) -> Tuple[str, List[Dict]]:
    
    # Initialize if empty
    if not history:
        history = initialize_conversation()
    
    history.append({"role": "user", "parts": [user_input]})
    
    try:
        generation_kwargs = {}
        if use_search:
            generation_kwargs["tools"] = [{"google_search": {}}]
        
        response = MODEL.generate_content(history, **generation_kwargs)
        agent_response = response.text
        history.append({"role": "model", "parts": [agent_response]})
        
        logger.info(f"Response generated — {len(agent_response)} chars")
        return agent_response, history
        
    except Exception as e:
        history.pop()
        error_msg = (
            f"❌ Hmm... Mere research servers mein thodi dikkat aa rahi hai.\n"
            f"Detail: {str(e)[:100]}\n"
            f"Kripya thodi der baad try karein ya 'reset' karke fresh start karein."
        )
        logger.error(f"API Error: {str(e)[:200]}")
        return error_msg, history


def get_help() -> str:
    return (
        "🔬 ProductGPT Capabilities:\n"
        "  🔍 Deep product research with specs analysis\n"
        "  ⚖️  Product comparisons (structured tables)\n"
        "  💡 Competitor alternatives & suggestions\n"
        "  📊 Value-for-money ratings\n"
        "  💰 Budget-based tier recommendations\n"
        "  🔄 'reset' — fresh context window\n"
        "  📊 'status' — session statistics\n"
        "  ❌ 'quit' — exit gracefully"
    )


# ─── QUICK TEST (FIXED) ───
print("🧪 Testing agent_think()...\n")

# ✅ FIX: Empty list, NOT None
test_history = []

test_response, test_history = agent_think("Hello, introduce yourself", test_history)

print(f"🤖 ProductGPT: {test_response[:150]}...\n")
print(f"📊 History messages: {len(test_history)}")
print(f"📊 Token estimate: {count_tokens(test_history):,}")
print("✅ Core logic operational!\n")

🧪 Testing agent_think()...



10:13:23 | INFO     | Response generated — 431 chars


🤖 ProductGPT: Namaste! Main hoon ProductGPT, aapka friendly aur expert product research assistant! 👋

Mera kaam hai aapki help karna sahi products choose karne mein...

📊 History messages: 4
📊 Token estimate: 259
✅ Core logic operational!



In [6]:
# ================================================================
# CELL 3: INTERACTION LAYER — MAIN AGENT LOOP
# ================================================================

def run_agent(enable_search: bool = False):
    """
    ProductGPT Main Interaction Loop.
    
    Args:
        enable_search: Enable Google Search grounding (default False for Week 2)
    """
    
    # ─── INITIALIZE ───
    history = initialize_conversation()
    state = AgentState()
    
    # ─── HEADER ───
    print("=" * 60)
    print("🔬  PRODUCTGPT — Expert Product Research Assistant")
    print("=" * 60)
    print("\n📋 Commands:")
    print("  • Product query  — 'laptop under 80000 for gaming'")
    print("  • help            — capabilities list")
    print("  • status          — session statistics")
    print("  • reset           — fresh conversation")
    print("  • quit / bye      — exit gracefully")
    print("\n" + "=" * 60)
    
    # ─── WELCOME ───
    print(f"\n🔬 ProductGPT: {WELCOME_MESSAGE}")
    print("\n" + "-" * 60)
    
    # ─── MAIN LOOP ───
    while True:
        try:
            # Get user input
            user_input = input("\n👤 You: ").strip()
            
            # ─── COMMAND: EXIT ───
            if user_input.lower() in ["quit", "bye", "exit", "stop"]:
                print("\n" + "=" * 60)
                print(f"👋 ProductGPT: Thanks for researching with me!")
                print(f"{'=' * 60}")
                print(f"📊 Session Summary:")
                print(f"   • Exchanges    : {state.exchange_count}")
                print(f"   • Duration     : {state.get_duration()}")
                print(f"   • Peak Tokens  : {state.peak_tokens:,}")
                print(f"   • Last Topic   : {state.current_topic or 'None'}")
                print(f"{'=' * 60}")
                break
            
            # ─── COMMAND: HELP ───
            if user_input.lower() == "help":
                print(f"\n{get_help()}")
                continue
            
            # ─── COMMAND: STATUS ───
            if user_input.lower() == "status":
                current_tokens = count_tokens(history)
                print(f"\n📊 Session Status:")
                print(f"   • Exchanges    : {state.exchange_count}")
                print(f"   • Duration     : {state.get_duration()}")
                print(f"   • Current Topic: {state.current_topic or 'None'}")
                print(f"   • Token Usage  : {current_tokens:,} / 1,000,000")
                print(f"   • Context Fill : {current_tokens / 10000:.1f}%")
                continue
            
            # ─── COMMAND: RESET ───
            if user_input.lower() in ["reset", "new chat", "fresh start"]:
                history = initialize_conversation()
                state.reset()
                print(f"\n🔄 Research session cleared!")
                print(f"🔬 ProductGPT: {WELCOME_MESSAGE}")
                continue
            
            # ─── ANTI-SPAM: EMPTY INPUT ───
            if not user_input:
                state.empty_input_count += 1
                if state.empty_input_count == 2:
                    print("🔬 ProductGPT: Kuch toh pucho — main research ke liye ready hoon! 🔍")
                elif state.empty_input_count >= 4:
                    print("🔬 ProductGPT: Lagta hai aap busy ho. 'quit' bolke exit kar sakte ho.")
                    state.empty_input_count = 0
                continue
            
            # Reset empty counter on valid input
            state.empty_input_count = 0
            
            # ─── AGENT PROCESSING ───
            print("🔬 Researching...", end=" ", flush=True)
            time.sleep(0.3)
            print("✓\n")
            
            agent_response, history = agent_think(
                user_input, 
                history, 
                use_search=enable_search
            )
            
            # Display response
            print(f"🔬 ProductGPT:\n{agent_response}")
            
            # ─── UPDATE STATE ───
            state.exchange_count += 1
            state.update_topic(user_input)
            
            # Track peak tokens
            current_tokens = count_tokens(history)
            state.peak_tokens = max(state.peak_tokens, current_tokens)
            
            print("\n" + "-" * 60)
            
        except KeyboardInterrupt:
            print(f"\n\n👋 ProductGPT: Session interrupted. Duration: {state.get_duration()}")
            break
            
        except Exception as e:
            logger.error(f"Loop error: {str(e)[:200]}")
            print(f"\n❌ Unexpected error. Type 'quit' to exit or try again.")
            continue


# ─── LAUNCH ───
if __name__ == "__main__":
    print("\n🎬 Starting ProductGPT...\n")
    run_agent(enable_search=False)  # Week 2: No search → No API error!


🎬 Starting ProductGPT...

🔬  PRODUCTGPT — Expert Product Research Assistant

📋 Commands:
  • Product query  — 'laptop under 80000 for gaming'
  • help            — capabilities list
  • status          — session statistics
  • reset           — fresh conversation
  • quit / bye      — exit gracefully


🔬 ProductGPT: Hello! Main hoon ProductGPT — aapka expert product research assistant. Kya research karna hai aaj? 🔍

------------------------------------------------------------



👤 You:  hello


🔬 Researching... ✓



10:13:29 | INFO     | Response generated — 111 chars


🔬 ProductGPT:
Namaste! 👋 Kaise hain aap? Batayein, aaj main aapki kya madad kar sakta hoon? Koi product research karna hai? 😊

------------------------------------------------------------



👤 You:  quit



👋 ProductGPT: Thanks for researching with me!
📊 Session Summary:
   • Exchanges    : 1
   • Duration     : 0m 10s
   • Peak Tokens  : 182
   • Last Topic   : None
